In [20]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [134]:
import numpy as np
import numpy.random as npr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pickle
from scipy import stats
from scipy.signal import convolve
from scipy.sparse import csr_matrix, vstack, hstack, issparse
import numpy as np
import copy
import statsmodels.api as sm
from imports import *
from config import dir_config
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from typing import Tuple, Dict, List
from tqdm import trange
import scipy.io
from scipy.sparse.linalg import lsqr


In [22]:
compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

In [92]:
def make_smooth_temporal_basis(filter_type, duration, center_spacing=50, n_bases=None, bin_size=1.0):
    """
    Create smooth temporal basis functions.

    Parameters
    ----------
    shape : str
        'raised cosine' or 'boxcar' (case-insensitive)
    duration : float
        Duration to cover (ms)
    center_spacing : int
        Spacing between basis function centers (ms)

    Returns
    -------
    'basis_matrix': ndarray,  # shape (n_time_bins, n_bases)

    """
    # number of bins in time (ensure integer)
    n_time_bins =  np.ceil(duration / bin_size).astype(int)
    if n_time_bins <= 0:
        raise ValueError("binfun(duration) must return a positive integer number of bins")

    if filter_type == "raised_cosine":
        n_bases = int(np.ceil(duration / center_spacing)) + 1
        time_bins = np.tile(np.arange(1, n_time_bins + 1).reshape(-1, 1), (1, n_bases)).astype(float)
        # spacing and width for raised cosines
        # center_spacing = n_time_bins / float(n_bases-1)
        bump_width = 4.0 * center_spacing
        centers = center_spacing * np.arange(n_bases, dtype=float)

        # compute x = time - center for each basis (broadcasting)
        x = time_bins - centers  # shape (n_time_bins, n_bases)

        # raised-cosine bump
        mask = np.abs(x / bump_width) < 0.5
        basis_matrix = np.where(mask, np.cos(x * 2.0 * np.pi / bump_width) * 0.5 + 0.5, 0.0)

    elif filter_type == "boxcar":
        # n_bases = int(np.ceil(duration / center_spacing))
        time_bins = np.tile(np.arange(1, n_time_bins + 1).reshape(-1, 1), (1, n_bases)).astype(float)

        # box_width = center_spacing
        box_width = n_time_bins / float(n_bases)
        basis_matrix = np.zeros_like(time_bins, dtype=float)
        centers = box_width * (np.arange(1, n_bases + 1, dtype=float)) - box_width / 2.0

        for k in range(n_bases):
            left = np.ceil(box_width * k)
            right = np.ceil(box_width * (k + 1))
            idx = (time_bins[:, k] > left) & (time_bins[:, k] <= right)
            count = idx.sum()
            if count > 0:
                basis_matrix[idx, k] = 1.0 / float(count)  # normalize

    else:
        raise ValueError(f"Unknown basis shape: {filter_type!r}")

    return basis_matrix


In [126]:
def delta_stim(event_times, n_timebins, values=None):
    """
    Create a sparse delta stimulus vector.

    Parameters
    ----------
    bin_times : array_like of int
        Event times (1-based indices, like MATLAB).
    n_timebins : int
        Total number of bins.
    values : array_like or None
        Values at each event time. Defaults to 1 for each event.

    Returns
    -------
    stim : scipy.sparse.csr_matrix, shape (n_timebins, 1)
        Sparse column vector with impulses at given times.
    """
    event_times = np.asarray(event_times, dtype=int) - 1
    # keep only events within timeline
    valid_mask = event_times <= n_timebins
    event_times = event_times[valid_mask]   # convert 1-based → 0-based

    if values is None:
        values = np.ones(len(event_times), dtype=float)
    else:
        assert len(values) == len(event_times), "Length of values must match number of event times"
        values = np.asarray(values, dtype=float)[valid_mask]



    # row indices = event times, col indices all 0 (single column vector)
    stim = csr_matrix((values, (event_times, np.zeros_like(event_times))),
                      shape=(n_timebins, 1))
    return stim

def boxcar_stim(start_bin, end_bin, n_timebins):
    """
    Create a boxcar stimulus design vector (sparse).

    Parameters
    ----------
    start_bin : int
        Start index
    end_bin : int
        End index
    n_timebins : int
        Total number of bins.

    Returns
    -------
    stim : scipy.sparse.csr_matrix, shape (n_timebins, 1)
        Sparse column vector with ones in the boxcar region.
    """
    # MATLAB is 1-based, Python is 0-based
    idx = np.arange(start_bin, end_bin)
    data = np.ones_like(idx, dtype=float)

    stim = csr_matrix((data, (idx, np.zeros_like(idx))), shape=(n_timebins, 1))
    return stim

In [ ]:
def compile_design_matrix(trial_indices, timepoints):

    trial_mats = []

    for trial in trial_indices:
        trial_timepoints = timepoints.loc[trial,:]
        conv_stim_on_matrix = stim_on(50, trial_timepoints)
        conv_target_on_matrix = target_on(trial_timepoints)
        conv_saccade_on_matrix = saccade_on(1, trial_timepoints)
        conv_bias_on_matrix = bias_on(1, trial_timepoints)
        miniX = hstack([conv_target_on_matrix, conv_stim_on_matrix, conv_saccade_on_matrix, conv_bias_on_matrix]).tocsr()
        trial_mats.append(csr_matrix(miniX))

    # Stack all trials vertically
    return vstack(trial_mats, format="csr")


In [97]:
def get_binned_spiketrain(trial_indices, spike_times, timepoints, bin_size=1.0):
    """
    Concatenate all trials' spike trains into a single sparse column vector.
    """
    spike_time_arrays = []
    total_trial_length = 0

    for trial in trial_indices:
        trial_start = timepoints.loc[trial, "target_onset"] - 50
        trial_end   = timepoints.loc[trial, "response_onset"]
        n_timebins  = int(np.ceil((trial_end - trial_start) / bin_size))

        # spikes relative to trial start
        trial_spike_times = spike_times[
            (spike_times >= trial_start) & (spike_times <= trial_end)
        ] - trial_start

        # convert to bin indices + shift into global timeline
        spike_time_arrays.append(trial_spike_times + total_trial_length)
        total_trial_length += n_timebins

    if len(spike_time_arrays) == 0:
        return csr_matrix((total_trial_length, 1))  # empty if no spikes

    # flatten
    spike_train_arrays = np.concatenate(spike_time_arrays)

    # make sparse binary spike train
    spike_train = delta_stim(spike_train_arrays, total_trial_length)

    return spike_train

In [98]:
def loss_neg_ll(weights, design_matrix, spike_train, reg_lambda=0.0):
    """
    Compute the negative log-likelihood loss for Poisson GLM with L2 regularization.

    Parameters
    ----------
    weights : ndarray, shape (n_features,)
        Model weights.
    design_matrix : ndarray or sparse matrix, shape (n_samples, n_features)
        Design matrix.
    spike_train : ndarray or sparse matrix, shape (n_samples,)
        Observed spike counts.
    reg_lambda : float
        Regularization strength.

    Returns
    -------
    loss_value : float
        Negative log-likelihood loss with regularization.
    """
    if issparse(design_matrix):
        design_matrix = design_matrix.toarray()
    if issparse(spike_train):
        spike_train = spike_train.toarray().ravel()

    linear_predictor = design_matrix @ weights
    rate = np.exp(linear_predictor)

    # Negative log-likelihood for Poisson
    nll = np.sum(rate) - np.sum(spike_train * linear_predictor)

    # L2 regularization term
    l2_reg = reg_lambda * np.sum(weights**2)

    return nll + l2_reg


In [131]:
def convolve_with_basis(event_matrix, basis_shape, basis_duration, n_bases=None):
    """
    Convolve event design matrix with temporal basis functions.

    Parameters
    ----------
    event_matrix : array-like or sparse, shape (n_timebins, n_events)
        Event indicator matrix (e.g. impulses or boxcars).
    basis_shape : str
        Basis type ('raised_cosine', 'boxcar', ...).
    basis_duration : int
        Duration of basis functions (ms).
    n_bases : int, optional
        Number of basis vectors. If None, use default from basis generator.

    Returns
    -------
    conv_matrix : csr_matrix, shape (n_timebins, n_events * n_bases)
        Convolved design matrix.
    """
    event_matrix = np.asarray(event_matrix.todense() if hasattr(event_matrix, "todense") else event_matrix)
    n_timebins, n_levels = event_matrix.shape

    B = make_smooth_temporal_basis(basis_shape, duration=basis_duration, n_bases=n_bases) # shape (basis_len, n_bases)
    n_bases = B.shape[1]

    conv_matrix = np.zeros((n_timebins, n_levels * n_bases))

    for e in range(n_levels):
        event_vector = event_matrix[:, e]
        for j in range(n_bases):
            conv_result = convolve(event_vector, B[:, j], mode="full")
            conv_matrix[:, e*n_bases + j] = conv_result[:n_timebins]
    return csr_matrix(conv_matrix)

def stim_on(coherence, timepoints, coh_levels = np.array([-50,-20,-6,0,6,20,50]), bin_size=1.0):
    duration = (timepoints["response_onset"] - timepoints["target_onset"] + 50)
    n_timebins = int(np.ceil(duration/ bin_size))
    coh_idx = np.where(coh_levels == coherence)[0]
    stim_on_matrix = np.zeros((n_timebins, len(coh_levels)))
    stim_on_matrix[:, coh_idx] = boxcar_stim(timepoints["stimulus_onset"], timepoints["response_onset"], n_timebins).toarray()
    return convolve_with_basis(stim_on_matrix, "raised_cosine", 500)

def target_on(timepoints, bin_size=1.0):
    duration = (timepoints["response_onset"] - timepoints["target_onset"] + 50)
    n_timebins = int(np.ceil(duration/ bin_size))
    target_on_matrix = delta_stim(timepoints["target_onset"], n_timebins)
    return convolve_with_basis(target_on_matrix, "raised_cosine", 200)

def saccade_on(choice, timepoints, choice_levels = np.array([0,1]), bin_size=1.0):
    duration = (timepoints["response_onset"] - timepoints["target_onset"] + 50)
    n_timebins = int(np.ceil(duration/ bin_size))
    choice_idx = np.where(choice_levels == choice)[0][0]
    saccade_on_matrix = np.zeros((n_timebins, len(choice_levels)))
    saccade_on_matrix[:, choice_idx] = delta_stim(timepoints["response_onset"], n_timebins).toarray().ravel()
    return convolve_with_basis(saccade_on_matrix, "raised_cosine", 1500)

def bias_on(bias, timepoints, bias_levels = np.array([0, 1]), bin_size=1.0):
    duration = (timepoints["response_onset"] - timepoints["target_onset"] + 50)
    n_timebins = int(np.ceil(duration/ bin_size))
    bias_idx = np.where(bias_levels == bias)[0][0]
    bias_on_matrix = np.zeros((n_timebins, len(bias_levels)))
    bias_on_matrix[:, bias_idx] = boxcar_stim(0, n_timebins-1, n_timebins).toarray().ravel()
    return convolve_with_basis(bias_on_matrix, "raised_cosine", 200)

# Pipeline
#### Event and variable matrix
- Target onset*DeltaStim, Stimulus Onset*BoxcarStim, Saccade*DeltaStim, Bias*BoxcarStim, Y(spiketrain between -50ms from target onset and Saccade)
#### Filters matrix
- Covariates: target_onset (200 ms, 5 bases), 7 * stimulus_onset (500 ms, 11 bases), 2 * Saccade (1500 ms, 31 bases; anti-causal), 2 * Bias (200 ms, 5 bases), post-spike history (10 1ms uniform basis + 10 raised cosine)
#### Convolve event and filter to get design matrix
#### Weighted design matrix to get fitted spike rate
#### Compare with actual spike rate and find best weights

In [ ]:
### Alternative (using statsmodels) ###
# model = sm.GLM(spike_train, X, family=sm.families.Poisson(sm.families.links.log()))
# results = model.fit()

In [100]:
session_to_exclude = ["210210_GP_JP","241209_GP_TZ"]

session_metadata = pd.read_csv(Path(processed_dir, 'sessions_metadata.csv'))
session_metadata = session_metadata[~np.isin(session_metadata["session_id"],session_to_exclude)]

neuron_metadata = pd.read_csv(Path(compiled_dir, 'neuron_metadata.csv'))
neuron_metadata = neuron_metadata[~np.isin(neuron_metadata["session_id"],session_to_exclude)]

with open(Path(processed_dir, f'glm_hmm_masked_final.pkl'), 'rb') as f:
    glm_hmm = pickle.load(f)


In [101]:
neuron_id = 113
session_name = neuron_metadata.loc[neuron_metadata["neuron_id"]==neuron_id, "session_id"].values[0]
data_path = Path(compiled_dir, session_name)

In [102]:
spike_times = np.load(data_path / "spike_times.npy")
spike_clusters = np.load(data_path / "spike_clusters.npy")
cluster_info = pd.read_csv(data_path / "cluster_info.tsv", sep="\t", index_col=False)

timestamps = pd.read_csv(Path(compiled_dir, session_name, f"{session_name}_timestamps.csv"), index_col=None)
trial_info = pd.read_csv(Path(compiled_dir, session_name, f"{session_name}_trial.csv"), index_col=None)

In [103]:
GP_trial_data = trial_info[trial_info.task_type == 1].reset_index(drop=True)

In [129]:
# get state (biased 1, unbiased 0)
confidence_threshold = 0.8
model = glm_hmm["model"]["models"][session_name]
choices = glm_hmm["data"][session_name]["choices"].values.reshape(-1, 1)
input = np.array(glm_hmm["data"][session_name][["normalized_stimulus", "bias", "prev_choice_1", "prev_target_1"]])
if glm_hmm["data"][session_name]["mask"] is None:
    mask = None
else:
    mask = glm_hmm["data"][session_name]["mask"]
mask = np.ones_like(choices, dtype=bool) if mask is None else mask

posterior_probs = model.expected_states(data=choices, input=input, mask=np.array(mask).reshape(-1, 1))[0]
biased_idx = (posterior_probs[:, 0] > confidence_threshold) & np.array(mask)
unbiased_idx = (posterior_probs[:, 1] > confidence_threshold) & np.array(mask)
GP_trial_data["state"] = np.nan
GP_trial_data.loc[np.where(biased_idx)[0], "state"] = 1
GP_trial_data.loc[np.where(unbiased_idx)[0], "state"] = 0

GP_trial_data = GP_trial_data[GP_trial_data.state.notna()]
GP_trial_data = GP_trial_data[GP_trial_data.reaction_time.notna()]
trial_indices = GP_trial_data.trial_number - 1  # zero-indexed


In [ ]:
timepoints = (timestamps / 30).round()
timepoints = timepoints.replace([np.inf, -np.inf], np.nan).astype("Int64")
start_time = timepoints["target_onset"] - 50
relative_timepoints = timepoints.sub(start_time, axis=0)


In [106]:
cluster_id = neuron_metadata.cluster[neuron_metadata["neuron_id"]==neuron_id].values[0]
neuron_spike_times = spike_times[spike_clusters == cluster_id]
neuron_spike_times = (neuron_spike_times / 30).round().astype(int)  # downsample to ms


In [ ]:
design_matrix = compile_design_matrix(trial_indices, relative_timepoints) # shape (total_time_bins, n_features)
# add bias term
bias = csr_matrix(np.ones((design_matrix.shape[0], 1)))
design_matrix = hstack([design_matrix, bias], format='csr')

spike_train = get_binned_spiketrain(trial_indices, neuron_spike_times, timepoints) # shape (total_time_bins, 1)

res = lsqr(design_matrix, spike_train.toarray().ravel())
# weights initialization: from least-squares solution
weight_initialization = res[0]  # shape (n_features,)

X = design_matrix.toarray() if issparse(design_matrix) else design_matrix
# X = sm.add_constant(X)  # add intercept term

res = minimize(
    fun=loss_neg_ll,
    x0=weight_initialization,
    method='L-BFGS-B',
    args=(design_matrix, spike_train, 0.1),
    options={"maxiter": 1000, "gtol": 1e-6, "disp": True}
)

fitted_weights = res.x          # MLE weights
neg_ll = res.fun      # negative log likelihood at optimum
exitflag = res.success